# 5. First Operations: Load, Inspect, and Display

### 🎯 Learning Goals

By the end of this notebook, you should be able to:

* Load a local image file using OpenCV and scikit-image.
* Use Matplotlib to correctly display grayscale and color images.
* Inspect image properties (`.shape`, `.dtype`, `.min()`, `.max()`).
* Access and manipulate individual pixels and channels using NumPy indexing.
* Convert a color image to grayscale.
* Plot and interpret an image's intensity histogram.

## Motivation

This is our first *practical* notebook. We'll take all the theory from the previous sections and put it into practice using our Python toolkit. We will perform the most fundamental operations in all of DIP.

In [3]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import cv2

# We will use the 'data' module from skimage to get standard test images
from skimage import data

%matplotlib inline

## 1. Loading and Displaying an Image

Let's load two standard test images from `skimage.data`: 
1.  `cameraman()`: A classic 8-bit grayscale image.
2.  `coffee()`: A standard 8-bit color image.

In [ ]:
# 1. Load a grayscale image
img_gray = data.camera()

# 2. Load a color image
# scikit-image loads images in the standard RGB format
img_rgb = data.coffee()

AttributeError: No skimage.data attribute cameraman

## 2. Inspecting Image Properties

Now that `img_gray` and `img_rgb` are in memory, let's check their properties. They are just NumPy arrays! 
This is the Python equivalent of Octave's `whos` or `size()` command (*PDI-Aula00Octave-1, Slide 11*).

* `.shape`: The dimensions (Height, Width) or (Height, Width, Channels).
* `.dtype`: The data type (e.g., `uint8` for 0-255).
* `.min()`, `.max()`: The minimum and maximum intensity values.

In [ ]:
print("--- Grayscale Image (Cameraman) ---")
print(f"Shape: {img_gray.shape}")
print(f"Data Type: {img_gray.dtype}")
print(f"Min Value: {img_gray.min()}, Max Value: {img_gray.max()}")
print(f"Total Pixels: {img_gray.size}")

print("\n--- Color Image (Coffee) ---")
print(f"Shape: {img_rgb.shape}")
print(f"Data Type: {img_rgb.dtype}")
print(f"Min Value: {img_rgb.min()}, Max Value: {img_rgb.max()}")
print(f"Total Pixels (H*W): {img_rgb.shape[0] * img_rgb.shape[1]}")

## 3. Displaying Images with Matplotlib

We use `plt.imshow()`. There are two important details:
1.  For **grayscale** images, you *must* specify `cmap='gray'` to tell Matplotlib to use a gray colormap. Otherwise, it will use a default "heatmap" and it will look strange.
2.  For **color** images, we don't need `cmap`, as Matplotlib correctly interprets the 3-channel shape.

We also add `plt.axis('off')` to hide the x and y axes.

In [ ]:
# Set up a figure with two subplots (1 row, 2 columns)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Display the grayscale image on the first axis
ax1.imshow(img_gray, cmap='gray')
ax1.set_title(f"Grayscale Image\nShape: {img_gray.shape}")
ax1.axis('off')

# Display the color image on the second axis
ax2.imshow(img_rgb)
ax2.set_title(f"Color (RGB) Image\nShape: {img_rgb.shape}")
ax2.axis('off')

plt.show()

## 4. Accessing Pixels & Splitting Channels

We can access pixels using NumPy's `[row, col]` indexing (remembering it's **0-indexed**). 

We can also use slicing to "split" the image into its R, G, and B channels. This is the Python version of Octave's `img(:,:,1)` syntax (*PDI-Aula00Octave-1, Slide 14*).

In [ ]:
# Get the pixel at row 100, column 50
pixel_g = img_gray[100, 50]
print(f"Grayscale pixel at [100, 50]: {pixel_g}")

# Get the pixel at row 200, column 250
pixel_rgb = img_rgb[200, 250]
print(f"Color pixel at [200, 250]: {pixel_rgb}")

# Get just the Green value from that pixel (channel 1)
pixel_green_val = img_rgb[200, 250, 1]
print(f"Green value at [200, 250]: {pixel_green_val}")

In [ ]:
# Split the R, G, B channels using slicing
# : means 'all values in this dimension'
R = img_rgb[:, :, 0] # Get all rows, all columns, channel 0 (Red)
G = img_rgb[:, :, 1] # Get all rows, all columns, channel 1 (Green)
B = img_rgb[:, :, 2] # Get all rows, all columns, channel 2 (Blue)

# Display the channels
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

ax1.imshow(R, cmap='gray')
ax1.set_title(f"Red Channel\nShape: {R.shape}")
ax1.axis('off')

ax2.imshow(G, cmap='gray')
ax2.set_title(f"Green Channel\nShape: {G.shape}")
ax2.axis('off')

ax3.imshow(B, cmap='gray')
ax3.set_title(f"Blue Channel\nShape: {B.shape}")
ax3.axis('off')

plt.show()

## 5. Converting to Grayscale

Converting from color to grayscale is a very common preprocessing step. It simplifies the image from 3 channels to 1, making many algorithms easier to apply. 

We can't just *average* the R, G, and B channels! Human eyes are more sensitive to Green light than Red or Blue. The standard conversion formula is a **weighted average**:

**`Y = 0.299*R + 0.587*G + 0.114*B`**

Luckily, we don't have to do this by hand. OpenCV's `cvtColor` function handles this for us. (This is the Python version of Octave's `rgb2gray` in *PDI-Aula00Octave-1, Slide 16*).

**Remember the BGR vs. RGB gotcha!**
1.  Our `img_rgb` is in RGB format (from `skimage`).
2.  OpenCV's `cvtColor` function expects BGR format by default for the `COLOR_BGR2GRAY` flag.
3.  Therefore, we must use the `COLOR_RGB2GRAY` flag instead!

In [ ]:
# Convert our RGB image to grayscale
img_coffee_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

# Display the original and the grayscale version side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

ax1.imshow(img_rgb)
ax1.set_title("Original Color (RGB)")
ax1.axis('off')

ax2.imshow(img_coffee_gray, cmap='gray')
ax2.set_title("Converted to Grayscale")
ax2.axis('off')

plt.show()

print(f"Original shape: {img_rgb.shape}, New shape: {img_coffee_gray.shape}")

## 6. Plotting the Intensity Histogram

A **histogram** is a graph that shows the distribution of pixel intensities in an image. 

* **X-axis:** The pixel intensity values (e.g., 0 to 255).
* **Y-axis:** The number of pixels in the image that have that intensity.

A histogram is a powerful tool. It tells us at a glance about the image's **contrast**:
* A **low-contrast** image will have all its pixels clustered in a narrow range (e.g., all values are between 80 and 150). 
* A **high-contrast** image will have its pixels spread out over the *entire* range, from dark (near 0) to bright (near 255).

We use `plt.hist()`. We must first `.ravel()` (or `.flatten()`) the 2D image array into a 1D list of pixels.

In [ ]:
# .ravel() flattens the 2D array into a 1D array
pixel_intensities = img_gray.ravel()

plt.figure(figsize=(10, 6))
# We specify 256 bins, one for each possible intensity value
plt.hist(pixel_intensities, bins=256, range=(0, 256), color='black', alpha=0.7)

plt.title("Intensity Histogram for 'Cameraman'")
plt.xlabel("Pixel Intensity (0=Black, 255=White)")
plt.ylabel("Number of Pixels")
plt.grid(True)
plt.show()

## 🧠 Interactive Check / Reflection

Look at the histogram above for the 'Cameraman' image.

1.  What does the tall spike near intensity `20-30` represent in the actual image? (Hint: Look at the `cameraman` image again. What's the darkest, largest object?)
2.  What about the large cluster of pixels between `100` and `200`?
3.  Does this image use the *full* intensity range (all the way to 255)? What does that tell you about its contrast?

## 📜 Summary

Congratulations! You have successfully performed the most common operations in DIP. You have:

* Loaded images using `skimage.data` (and we know how to use `cv2.imread`).
* Inspected their `shape` and `dtype`.
* Displayed them with `plt.imshow()`, using `cmap='gray'` for grayscale.
* Accessed pixels and split channels using NumPy `[row, col, chan]` slicing.
* Converted from RGB to Grayscale using `cv2.cvtColor()`.
* Plotted and analyzed an intensity histogram with `plt.hist()`.

## 🚀 Next Steps

This notebook completes our introductory module. We have a solid foundation. 

Next, in **Module 2**, we will dive deep into that histogram, learning how to manipulate it to perform **Intensity Transformations and Histogram Equalization** to dramatically improve image contrast.